# Neo4j问答运行测试

## 导入引用

In [1]:
import os
from neo4j_driver import Neo4jConnection, Node
from answer_search import AnswerSearcher
from question_parser import QuestionPaser
from question_classifier import QuestionClassifier

## 实例化对象

In [2]:
g = Neo4jConnection('bolt://localhost:7687/', 'admin', '73@TuGraph')
classifier = QuestionClassifier()
parser = QuestionPaser()
searcher = AnswerSearcher(g)

## 输入问题

In [3]:
question = '癫痫有什么症状'

## 从问题中提取实体和关系

In [4]:
res_classify = classifier.classify(question)

In [5]:
res_classify

{'args': {'癫痫': ['disease']}, 'question_types': ['disease_symptom']}

## 问题解析-生成Cypher查询语句

In [6]:
res_cypher = parser.parser_main(res_classify)

In [7]:
res_cypher

[{'question_type': 'disease_symptom',
  'sql': ["MATCH (m:Disease)-[r:has_symptom]->(n:Symptom) where m.name = '癫痫' return m.name, r.name, n.name"]}]

## 生成回答

In [4]:
question = '癫痫怎么检查'
res_classify = classifier.classify(question)
res_cypher = parser.parser_main(res_classify)
searcher.getreply(res_cypher)


贝美格诱发试验 头颅平片 MRI 脑电图 脑血流灌注断层显像 [详细]


In [5]:
question = '癫痫有什么症状'
res_classify = classifier.classify(question)
res_cypher = parser.parser_main(res_classify)
searcher.getreply(res_cypher)

四肢抽搐
昏睡
一过性昏厥
反复高热
惊厥


In [6]:
result = g.query(res_cypher[0]['sql'][0])
result

[<Record m.name='癫痫' r.name='症状' n.name='四肢抽搐'>,
 <Record m.name='癫痫' r.name='症状' n.name='昏睡'>,
 <Record m.name='癫痫' r.name='症状' n.name='一过性昏厥'>,
 <Record m.name='癫痫' r.name='症状' n.name='反复高热'>,
 <Record m.name='癫痫' r.name='症状' n.name='惊厥'>]

四肢抽搐
昏睡
一过性昏厥
反复高热
惊厥
